# Gavety AI worker (pull mode)
Dev/demo worker. Kaggle sessions are time-limited batch notebooks, not servers: stop = no analysis. Check Kaggle's current terms before relying on this, and only send footage you are allowed to share (frames leave your site).

**Setup once:** (1) zip the `worker/gavety_worker` folder + your `config.json` and upload as a *private* Kaggle dataset named `gavety-worker`; add it to this notebook. (2) Notebook settings: Internet ON, GPU ON. (3) Add secret `GAVETY_WORKER_TOKEN` (Add-ons > Secrets).

In [ ]:
!pip -q install ultralytics requests pillow numpy

In [ ]:
import os, sys, json, shutil, glob
src = glob.glob('/kaggle/input/gavety-worker*')[0]
shutil.copytree(src, '/kaggle/working/worker', dirs_exist_ok=True)
os.chdir('/kaggle/working/worker'); sys.path.insert(0, '.')
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret('GAVETY_WORKER_TOKEN')
HUB = 'https://YOUR-BROKER-HOST'   # <- your broker URL
import torch; print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

In [ ]:
import time
from gavety_worker.engines import build_engines
from gavety_worker.pull_worker import pull_loop
engines = build_engines(json.load(open('config.json')))
END = time.time() + 3*3600   # stop after 3 h so the session ends cleanly
n = pull_loop(HUB, TOKEN, engines, worker_id='kaggle', stop=lambda: time.time() > END)
print('handled', n, 'frames')